# Exp33 — Freeze the final evaluation manifests for PRR and its controls

## Research goal

This notebook freezes a single deterministic evaluation protocol shared by the conventional **Pearson** retrieval baseline, the main **PRR** method, and the restricted **PRR-Stat** ablation.

The paper separates two questions:

1. **Retrieval relevance** — does PRR retrieve histories with futures more compatible with the query future than Pearson?
2. **Forecast utility** — when the resulting historical forecast is combined with the same frozen direct forecaster, does it help under validation-selected trust?

PRR-Stat is retained only as a fixed-support ablation that isolates future-supervised reranking within Pearson support before full PRR adds learned embedding relations and candidate-support expansion.

## Why this notebook

Historical runs did not always persist standalone `anchor × channel` manifests. Rather than mix query sets across methods, this experiment freezes one **new, deterministic, channel-balanced evaluation manifest** and reuses it for every retrieval method and forecasting backbone.

## Fixed protocol

- datasets: ETTh1, Weather, Electricity, Traffic, Exchange, Solar
- horizons: 96, 192, 336, 720
- validation pairs: 4,096
- test pairs: 8,192 (6,384 for Exchange H=720 when the admissible set is smaller)
- channel-balanced deterministic sampling
- chronological splits and horizon-aware anchors
- same query pairs for Pearson, PRR-Stat, PRR, and all frozen forecasters
- no model training in this notebook

In [ ]:

# Cell 1 — Configuration

from pathlib import Path
import json, hashlib, os, re
import numpy as np
import pandas as pd

ROOT = Path(os.environ.get("PRR_ROOT", str(Path.cwd().resolve().parents[1] if Path.cwd().name == "prr_long_horizon" else Path.cwd().resolve())))
EXP32_ROOT = ROOT / "results" / "exp32"

RESULT_ROOT = ROOT / "results" / "exp33"
MANIFEST_ROOT = RESULT_ROOT / "final_eval_manifest"
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)

DATASETS = ("ETTh1","Weather","Electricity","Traffic","Exchange","Solar")
HORIZONS = (96,192,336,720)

VAL_PAIRS = 4096
TEST_PAIRS = 8192
SEQ_LEN = 336
MIN_LOOKBACK = 512

# Frozen before final PRR-Stat vs PRR comparison.
EVAL_SEED_BASE = 21100

# We use large separated offsets only to make every dataset reproducible and independent.
# These are protocol identifiers, not tuned hyperparameters.
DATASET_SEED_OFFSET = {
    "ETTh1": 0,
    "Weather": 1000,
    "Electricity": 2000,
    "Traffic": 3000,
    "Exchange": 4000,
    "Solar": 5000,
}

SPLIT_SEED_EXTRA = {
    "val": 100,
    "test": 200,
}

EXPECTED_CHANNELS = {
    "ETTh1": 7,
    "Weather": 21,
    "Electricity": 321,
    "Traffic": 862,
    "Exchange": 8,
    "Solar": 137,
}

print("RESULT_ROOT  :", RESULT_ROOT)
print("MANIFEST_ROOT:", MANIFEST_ROOT)
print("Seed base    :", EVAL_SEED_BASE)
print("Val/Test pairs:", VAL_PAIRS, TEST_PAIRS)


In [ ]:

# Cell 2 — Resolve the exact six raw datasets
# Note Solar uses capital-S directory in the historical codebase.

DATA_CANDIDATES = {
    "ETTh1": [
        Path("/data/Time-Series-Library/dataset/ETT-small/ETTh1.csv"),
        Path("/data/Time-Series-Library_v2/dataset/ETT-small/ETTh1.csv"),
        Path("/data/dataset/ETTh1.csv"),
    ],
    "Weather": [
        Path("/data/Time-Series-Library/dataset/weather/weather.csv"),
        Path("/data/Time-Series-Library_v2/dataset/weather/weather.csv"),
        Path("/data/dataset/weather/weather.csv"),
        Path("/data/dataset/weather.csv"),
    ],
    "Electricity": [
        Path("/data/Time-Series-Library/dataset/electricity/electricity.csv"),
        Path("/data/Time-Series-Library_v2/dataset/electricity/electricity.csv"),
        Path("/data/dataset/electricity/electricity.csv"),
        Path("/data/dataset/electricity.csv"),
    ],
    "Traffic": [
        Path("/data/Time-Series-Library/dataset/traffic/traffic.csv"),
        Path("/data/Time-Series-Library_v2/dataset/traffic/traffic.csv"),
        Path("/data/dataset/traffic/traffic.csv"),
        Path("/data/dataset/traffic.csv"),
    ],
    "Exchange": [
        Path("/data/Time-Series-Library/dataset/exchange_rate/exchange_rate.csv"),
        Path("/data/Time-Series-Library_v2/dataset/exchange_rate/exchange_rate.csv"),
        Path("/data/dataset/exchange_rate/exchange_rate.csv"),
        Path("/data/dataset/exchange_rate.csv"),
        Path("/data/dataset/exchange.csv"),
    ],
    "Solar": [
        Path("/data/Time-Series-Library/dataset/Solar/solar_AL.txt"),
        Path("/data/Time-Series-Library_v2/dataset/Solar/solar_AL.txt"),
        Path("/data/dataset/Solar/solar_AL.txt"),
        Path("/data/dataset/solar_AL.txt"),
    ],
}

dataset_paths = {}

for d, candidates in DATA_CANDIDATES.items():
    p = next((x for x in candidates if x.exists()), None)
    dataset_paths[d] = p
    print(f"{d:11s} -> {p}")

missing = [d for d,p in dataset_paths.items() if p is None]
assert not missing, f"Missing raw datasets: {missing}"

print("\n[PASS] 6/6 raw datasets resolved.")


In [ ]:

# Cell 3 — Load only dataset shape and establish chronological split boundaries

def load_raw_shape(dataset, path):
    if dataset == "Solar":
        # Historical Solar loader: CSV first, whitespace fallback.
        try:
            df = pd.read_csv(path, header=None)
            if df.shape[1] == 1:
                df = pd.read_csv(path, header=None, sep=r"\s+")
        except Exception:
            df = pd.read_csv(path, header=None, sep=r"\s+")
        return len(df), df.shape[1]

    df = pd.read_csv(path, nrows=None)
    c = df.shape[1] - (1 if "date" in df.columns else 0)
    return len(df), c

def split_bounds(dataset, n):
    # ETTh1: paper-fixed 8640/11520/rest boundaries (test continues to the end of the file).
    if dataset == "ETTh1":
        train_end = 12 * 30 * 24      # 8640
        val_end   = train_end + 4 * 30 * 24  # 11520
        test_end  = n
    else:
        train_end = int(0.70 * n)
        num_test  = int(0.20 * n)
        val_end   = n - num_test
        test_end  = n

    return int(train_end), int(val_end), int(test_end)

dataset_meta = {}

for d in DATASETS:
    n, c = load_raw_shape(d, dataset_paths[d])
    assert c == EXPECTED_CHANNELS[d], f"{d}: channels={c}, expected={EXPECTED_CHANNELS[d]}"
    tr, va, te = split_bounds(d, n)

    dataset_meta[d] = {
        "rows": n,
        "channels": c,
        "train_end": tr,
        "val_end": va,
        "test_end": te,
        "path": str(dataset_paths[d]),
    }

meta_df = pd.DataFrame([
    {"dataset": d, **m}
    for d,m in dataset_meta.items()
])

display(meta_df)
meta_df.to_csv(RESULT_ROOT / "dataset_split_manifest.csv", index=False)

print("[PASS] dataset shapes and chronological boundaries frozen.")


In [ ]:

# Cell 4 — Deterministic channel-balanced pair sampler
#
# This follows the same sampler FAMILY as the earlier experiments:
# 1) horizon-aware valid anchors
# 2) balanced pair budget across channels
# 3) random anchor draw within each channel
# 4) lexicographic sort by anchor then channel

def eval_anchors(start, end, horizon, stride=1, lookback=512):
    lo = max(int(start), int(lookback))
    hi = int(end) - int(horizon)

    if hi < lo:
        return np.empty(0, dtype=np.int64)

    return np.arange(
        lo,
        hi + 1,
        int(stride),
        dtype=np.int64,
    )

def sample_eval_pairs(dataset, horizon, split, n_pairs):
    m = dataset_meta[dataset]

    if split == "val":
        start = m["train_end"]
        end = m["val_end"]
    elif split == "test":
        start = m["val_end"]
        end = m["test_end"]
    else:
        raise ValueError(split)

    anchors = eval_anchors(
        start,
        end,
        horizon,
        stride=1,
        lookback=max(MIN_LOOKBACK, SEQ_LEN),
    )

    C = int(m["channels"])

    n_pairs = min(
        int(n_pairs),
        len(anchors) * C,
    )

    base = n_pairs // C
    rem = n_pairs % C

    seed = (
        EVAL_SEED_BASE
        + DATASET_SEED_OFFSET[dataset]
        + SPLIT_SEED_EXTRA[split]
    )

    rng = np.random.default_rng(seed)

    extra_channels = set(
        rng.choice(
            C,
            size=rem,
            replace=False,
        ).tolist()
        if rem > 0 else []
    )

    aa_list = []
    cc_list = []

    for ch in range(C):
        n_ch = base + (1 if ch in extra_channels else 0)
        if n_ch <= 0:
            continue

        aa = rng.choice(
            anchors,
            size=n_ch,
            replace=(n_ch > len(anchors)),
        )

        aa_list.append(aa.astype(np.int64))
        cc_list.append(
            np.full(n_ch, ch, dtype=np.int64)
        )

    pair_anchor = np.concatenate(aa_list)
    pair_channel = np.concatenate(cc_list)

    order = np.lexsort(
        (pair_channel, pair_anchor)
    )

    return (
        pair_anchor[order],
        pair_channel[order],
        seed,
        len(anchors),
    )


In [ ]:

# Cell 5 — Generate and freeze all 48 manifests

rows = []

for dataset in DATASETS:
    for h in HORIZONS:
        for split, n_target in [
            ("val", VAL_PAIRS),
            ("test", TEST_PAIRS),
        ]:
            a, c, seed, n_valid_anchors = sample_eval_pairs(
                dataset,
                h,
                split,
                n_target,
            )

            out = MANIFEST_ROOT / f"{dataset}_H{h}_{split}.npz"

            np.savez_compressed(
                out,
                anchor=a.astype(np.int64),
                channel=c.astype(np.int16),
            )

            # Stable digest for reproducibility / accidental modification detection.
            digest = hashlib.sha256(
                a.astype(np.int64).tobytes()
                + c.astype(np.int16).tobytes()
            ).hexdigest()

            rows.append({
                "dataset": dataset,
                "horizon": h,
                "split": split,
                "target_pairs": n_target,
                "actual_pairs": len(a),
                "unique_anchors": len(np.unique(a)),
                "channels_used": len(np.unique(c)),
                "channel_min": int(c.min()),
                "channel_max": int(c.max()),
                "anchor_min": int(a.min()),
                "anchor_max": int(a.max()),
                "valid_anchor_pool": n_valid_anchors,
                "seed": seed,
                "sha256": digest,
                "path": str(out),
            })

manifest_df = pd.DataFrame(rows).sort_values(
    ["dataset","horizon","split"]
).reset_index(drop=True)

display(manifest_df)

assert len(manifest_df) == 48
assert (manifest_df.actual_pairs > 0).all()

for d in DATASETS:
    dd = manifest_df[manifest_df.dataset.eq(d)]
    assert (dd.channels_used == EXPECTED_CHANNELS[d]).all()

manifest_df.to_csv(
    RESULT_ROOT / "final_eval_manifest_index.csv",
    index=False,
)

print("\n[PASS] 48/48 deterministic final manifests created.")


In [ ]:

# Cell 6 — Compare with any saved Exp32 manifests as an AUDIT ONLY
#
# Exact equality is not required because Exp33 intentionally freezes
# one common final protocol.  The old artifacts are not allowed to
# determine method-specific query sets.

audit_rows = []

old_index = EXP32_ROOT / "pair_manifest_selected_preflight.csv"

if old_index.exists():
    old = pd.read_csv(old_index)

    for _, r in old[old.status.eq("FOUND")].iterrows():
        d = str(r.dataset)
        h = int(r.horizon)
        sp = str(r["split"])

        new_path = MANIFEST_ROOT / f"{d}_H{h}_{sp}.npz"

        if not new_path.exists():
            continue

        with np.load(new_path, allow_pickle=False) as z:
            new_pairs = set(
                zip(
                    z["anchor"].astype(np.int64).tolist(),
                    z["channel"].astype(np.int64).tolist(),
                )
            )

        old_path = Path(str(r.path))
        if not old_path.exists():
            audit_rows.append({
                "dataset": d, "horizon": h, "split": sp,
                "old_exists": False,
            })
            continue

        try:
            if old_path.suffix.lower() == ".npz":
                with np.load(old_path, allow_pickle=False) as z:
                    keys = {k.lower():k for k in z.files}
                    ak = next((keys[k] for k in ("anchor","pair_anchor","query_anchor") if k in keys), None)
                    ck = next((keys[k] for k in ("channel","pair_channel","query_channel","candidate_channel") if k in keys), None)

                    if ak is None or ck is None:
                        continue

                    oa = np.asarray(z[ak]).reshape(-1).astype(np.int64)
                    oc = np.asarray(z[ck]).reshape(-1).astype(np.int64)

            elif old_path.suffix.lower() == ".csv":
                df = pd.read_csv(old_path)
                low = {str(k).lower():k for k in df.columns}
                ak = next((low[k] for k in ("anchor","pair_anchor","query_anchor") if k in low), None)
                ck = next((low[k] for k in ("channel","pair_channel","query_channel","candidate_channel") if k in low), None)
                if ak is None or ck is None:
                    continue
                oa = df[ak].to_numpy(dtype=np.int64)
                oc = df[ck].to_numpy(dtype=np.int64)
            else:
                continue

            old_pairs = set(zip(oa.tolist(), oc.tolist()))
            inter = len(new_pairs & old_pairs)
            union = len(new_pairs | old_pairs)

            audit_rows.append({
                "dataset": d,
                "horizon": h,
                "split": sp,
                "old_exists": True,
                "old_n": len(oa),
                "new_n": len(new_pairs),
                "intersection": inter,
                "jaccard": inter / union if union else np.nan,
                "exact_set_equal": old_pairs == new_pairs,
            })

        except Exception as e:
            audit_rows.append({
                "dataset": d,
                "horizon": h,
                "split": sp,
                "old_exists": True,
                "error": repr(e),
            })

audit_df = pd.DataFrame(audit_rows)

if len(audit_df):
    display(audit_df)
    audit_df.to_csv(
        RESULT_ROOT / "old_vs_final_manifest_audit.csv",
        index=False,
    )
else:
    print("No old manifest audit rows available.")

print(
    "\n[NOTE] Old-manifest overlap is diagnostic only. "
    "PRR-Stat and PRR will both use the new Exp33 manifests."
)


In [ ]:

# Cell 7 — Freeze the aligned method-comparison contract

contract = {
    "experiment": "Exp33",
    "purpose": "aligned_common_evaluation_manifest",
    "datasets": list(DATASETS),
    "horizons": list(HORIZONS),
    "validation_pairs": VAL_PAIRS,
    "test_pairs": TEST_PAIRS,
    "seq_len_forecaster": SEQ_LEN,
    "minimum_history_for_eval_anchor": MIN_LOOKBACK,
    "eval_seed_base": EVAL_SEED_BASE,
    "dataset_seed_offset": DATASET_SEED_OFFSET,
    "split_seed_extra": SPLIT_SEED_EXTRA,
    "methods_primary": [
        "Direct",
        "PRR-Stat",
        "PRR",
    ],
    "method_optional_secondary": "PRR-Stat (ablation only)",
    "retrieval_primary_metrics": [
        "AnalogFutureMSE",
        "Recall@K",
        "NDCG@K",
    ],
    "downstream_metrics": [
        "MSE",
        "MAE",
    ],
    "interpretation_rule": (
        "PRR is evaluated first as a retrieval improvement. "
        "Downstream forecasting may improve, tie, or degrade relative to PRR-Stat; "
        "all outcomes are reported."
    ),
    "manifest_root": str(MANIFEST_ROOT),
}

contract_path = RESULT_ROOT / "final_evaluation_contract.json"
contract_path.write_text(
    json.dumps(contract, indent=2),
    encoding="utf-8",
)

print(json.dumps(contract, indent=2))

print("\n" + "="*100)
print("EXP33 COMPLETE")
print("="*100)
print("48/48 final query manifests are frozen.")
print("NEXT:")
print("1) Generate PatchTST direct caches on these exact manifests.")
print("2) Evaluate PRR-Stat and PRR on the SAME manifests.")
print("3) Compare retrieval first, then frozen-forecaster MSE/MAE.")
print("4) Use PRR-Stat only as the restricted fixed-support ablation.")
